# Pályázati Felhívás Kockázati Elemző Pipeline

Ez a notebook bemutatja a teljes ML pipeline működését lépésről lépésre:

1. **PDF szövegkinyerés** – pdfplumber
2. **ZSC kategorizálás** – Zero-Shot Classification (facebook/bart-large-mnli)
3. **Intent felismerés** – TF-IDF + Logistic Regression
4. **Kockázati pontozás** – szabály-alapú modell
5. **LLM összefoglaló** – TinyLlama / Ollama
6. **Monitoring KPI-k** – adatminőség és drift detekció

## 0. Környezet beállítás

In [ ]:
import sys
from pathlib import Path

# Projekt gyökér hozzáadása a Python pathhoz
BASE_DIR = Path.cwd()
if BASE_DIR.name == 'notebooks':
    BASE_DIR = BASE_DIR.parent

if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

print('✅ BASE_DIR:', BASE_DIR)
print('✅ Python path beállítva')

In [ ]:
# Függőségek importálása
import pandas as pd
import sqlite3

from app.extractor import read_pdf_text, extract_fields
from app.zsc_classifier import classify_text, LABELS
from app.intent_model import IntentRecognizer
from app.risk_model import compute_risk
from app.llm_summary import generate_llm_indicator_material, PROMPT_VERSION
from app.monitoring import compute_kpis

print('✅ Összes modul sikeresen importálva')
print(f'📝 Aktív prompt verzió: {PROMPT_VERSION}')

## 1. Training Set bemutatása

### 1.1 ZSC – Zero-Shot Classification
- **Modell**: `facebook/bart-large-mnli` (HuggingFace)
- **Training set**: nincs – nulla-shot megközelítés, csak a label-ek neve számít
- **Label-ek**: lásd alább

### 1.2 Intent Recognizer
- **Modell**: TF-IDF vektorizátor + Logistic Regression
- **Training set**: 25 kézzel annotált magyar mondat (5 osztály × 5 példa)
- **Újratanítás**: ha új osztályok vagy mintaszövegek szükségesek, az `IntentRecognizer._train()` metódus bővítendő

In [ ]:
# ZSC label-ek bemutatása
print('ZSC kategóriák (zero-shot, nincs tanítóadat):')
for i, label in enumerate(LABELS, 1):
    print(f'  {i}. {label}')

print()

# Intent training set bemutatása
intent_train = {
    'summary':            ['Foglald össze ezt a pályázati felhívást', 'Mi a felhívás lényege'],
    'indicator_analysis': ['Milyen indikátorok vannak a felhívásban', 'Elemezd az indikátorokat'],
    'financial_info':     ['Mekkora a támogatás összege', 'Mennyi az előleg mértéke'],
    'eligibility':        ['Kik pályázhatnak', 'Ki lehet kedvezményezett'],
    'risk_analysis':      ['Milyen kockázatok vannak', 'Kockázatértékelést kérek'],
}

df_train = pd.DataFrame([
    {'Osztály': cls, 'Példa': example}
    for cls, examples in intent_train.items()
    for example in examples
])
print('Intent training set minta:')
print(df_train.to_string(index=False))

## 2. PDF feldolgozás – szövegkinyerés

In [ ]:
PDF_DIR = BASE_DIR / 'data' / 'pdfs'
pdf_files = sorted(PDF_DIR.glob('*.pdf'))

print(f'📁 Talált PDF fájlok: {len(pdf_files)}')
for f in pdf_files:
    print(f'  - {f.name}')

In [ ]:
if not pdf_files:
    print('⚠️  Nincs PDF fájl a data/pdfs/ mappában.')
else:
    sample_pdf = pdf_files[0]
    print(f'📄 Tesztfájl: {sample_pdf.name}')

    text = read_pdf_text(str(sample_pdf))
    print(f'📝 Kinyert szöveg hossza: {len(text)} karakter')
    print()
    print('--- Első 500 karakter ---')
    print(text[:500])

In [ ]:
if pdf_files:
    fields = extract_fields(text)
    print('🔍 Kinyert strukturált mezők:')
    df_fields = pd.DataFrame(list(fields.items()), columns=['Mező', 'Érték'])
    print(df_fields.to_string(index=False))

## 3. ZSC – Zero-Shot Classification (HuggingFace pipeline)

In [ ]:
# FIGYELEM: Ez a cella letölti a facebook/bart-large-mnli modellt (~1.6 GB)
# Első futtatáskor hosszabb időt vehet igénybe.

if pdf_files:
    print('🤖 ZSC fut...')
    zsc_category = classify_text(text[:1500])
    print(f'✅ ZSC kategória: {zsc_category}')

## 4. Intent Recognizer (TF-IDF + Logistic Regression)

In [ ]:
recognizer = IntentRecognizer()
print('✅ Intent modell betanítva')

# Tesztek
test_queries = [
    'Foglald össze a felhívást',
    'Milyen kockázatok vannak',
    'Mekkora a támogatás összege',
    'Kik pályázhatnak',
    'Milyen indikátorok szerepelnek',
]

print('\nTeszt predikciók:')
for q in test_queries:
    intent = recognizer.predict(q)
    proba = recognizer.predict_proba(q)
    top_prob = max(proba.values())
    print(f'  "{q}" → {intent} ({top_prob:.0%})')

In [ ]:
if pdf_files:
    intent = recognizer.predict(text[:2000])
    print(f'📄 PDF alapú intent: {intent}')

    # Modell mentése
    recognizer.save()
    print('💾 Modell elmentve: models/ mappába')

## 5. Kockázati modell (szabály-alapú)

In [ ]:
if pdf_files:
    risk_score, risk_category = compute_risk(fields)
    print(f'⚠️  Kockázati pontszám: {risk_score}')
    print(f'🏷️  Kockázati kategória: {risk_category}')

    # Pontszám magyarázata
    print('\nPontszám összetevők:')
    items = [
        ('Előleg %', fields.get('advance_percent')),
        ('Projekt időtartam (hónap)', fields.get('project_duration_months')),
        ('Max támogatás (Ft)', fields.get('max_support')),
        ('Projektek száma', fields.get('project_count')),
        ('Támogatás típusa', fields.get('support_type')),
        ('Önerő szükséges', fields.get('own_fund_required')),
        ('Konzorcium', fields.get('consortium_allowed')),
    ]
    for name, val in items:
        print(f'  {name}: {val}')

## 6. LLM összefoglaló (TinyLlama / Ollama)

**Prompt verziózás**: a `llm_summary.py` fájlban `PROMPT_VERSION` konstans rögzíti az aktuális verziót.  
Minden prompt változtatást git commit-tal kell dokumentálni a következő sablon szerint:
```
git commit -m 'prompt(v3): rövidebb összefoglaló, kockázati hangsúly növelve'
```

In [ ]:
if pdf_files:
    print(f'📝 Prompt verzió: {PROMPT_VERSION}')
    print('🤖 LLM összefoglaló generálása (TinyLlama fallback ha Ollama nem elérhető)...')

    llm_output = generate_llm_indicator_material(text, model='tinyllama')

    print('\n--- LLM kimenet ---')
    print(llm_output)

## 7. Monitoring és KPI-k

In [ ]:
kpis = compute_kpis()

print('📊 Pipeline KPI-k:')
print(f"  Összes rekord: {kpis.get('total_grants', 0)}")
print(f"  LLM hiányosság arány: {kpis.get('llm_missing_rate', 'n/a')}")
print(f"  Előleg adat hiány: {kpis.get('advance_missing_rate', 'n/a')}")
print(f"  Hiányzó felhívás kód: {kpis.get('call_code_missing', 'n/a')}")

if 'risk_distribution' in kpis:
    print('\nKockázati megoszlás:')
    for k, v in kpis['risk_distribution'].items():
        bar = '█' * v
        print(f'  {k:12s}: {bar} ({v})')

if 'zsc_distribution' in kpis:
    print('\nZSC kategória megoszlás:')
    for k, v in kpis['zsc_distribution'].items():
        print(f'  {k:30s}: {v}')

In [ ]:
# Data drift detekció – mezők kitöltöttsége
DB_PATH = BASE_DIR / 'grants.db'

if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    df = pd.read_sql_query('SELECT * FROM grants', conn)
    conn.close()

    if not df.empty:
        print('🔍 Mezők kitöltöttsége (data drift indikátor):')
        fill_rate = (df.notna() & (df != '') & (df != 'nincs adat')).mean() * 100
        for col, rate in fill_rate.sort_values().items():
            status = '✅' if rate > 80 else ('⚠️' if rate > 40 else '❌')
            bar = '█' * int(rate / 10)
            print(f'  {status} {col:30s}: {bar} {rate:.0f}%')
    else:
        print('ℹ️  Adatbázis üres. Futtasd: python -m app.ingest')
else:
    print('ℹ️  grants.db nem található. Futtasd: python -m app.ingest')

## 8. Teljes pipeline futtatása (batch)

In [ ]:
# A teljes ingest pipeline futtatása:
# Figyelem: le kell töltenie a modelleket (~1.6 GB bart + ~600 MB TinyLlama)

run_full = False  # Állítsd True-ra, ha futtatni szeretnéd

if run_full:
    from app.ingest import run
    run()
    print('\n✅ Pipeline lefutott')
    kpis_after = compute_kpis()
    print(f"Feldolgozott rekordok: {kpis_after.get('total_grants', 0)}")
else:
    print('ℹ️  Pipeline futtatás kikapcsolva (run_full = False)')
    print('   Állítsd True-ra a fenti változót a futtatáshoz.')